# Imports

In [2]:
# ------------------------
# Imports and Setup
# ------------------------
import os,glob,sys
import math
import random
import numpy as np
from scipy import stats

import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path

# Load this for CTAE two region
from models.ctae import CoupledTransformerAutoencoderTwoRegions 

# Load this for CTAE multi region
from models.ctae import CoupledTransformerAutoencoderMultiRegion

from utils_correct import create_train_val_loaders, train_ctae_with_logging, safe_format, create_test_loader

# Usage

## setup

In [3]:

# Your data are LFP windows, not binned spikes
fs = 500          # after downsampling
window_sec = 0.5
bin_size = 1 / fs # only used if you need a time vector

# Transformer settings: start small
nhead = 1
num_layers = 2

learning_rate = 1e-4

# Start conservative
lambda_ortho = 1e-3
lambda_alignment = 0.05
lambda_recons2 = 1
lambda_shared = 1

warm_up_ortho = 20

# Positional encoding
pe = True
pe_learn = False

# Your windows are ~247 or 250 timepoints
max_len = 300

batch_size = 8   # or 16 if GPU memory allows
num_epochs = 300 # start with 300, not 5000

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


cuda:0
NVIDIA RTX 5000 Ada Generation


In [4]:
# if os.path.exists(model_path):
#     print(f"Model already exists: {model_path}")
# else:
DATA_ROOT = r"D:\copy_comp_project\Off_tensor_Data_L"#################
TRAINED_MODELS_ROOT = r"D:\copy_comp_project\ctae_models"
os.makedirs(TRAINED_MODELS_ROOT, exist_ok=True)
subject_list = ["s508", "s513","s514", "s515","s518","s519","s520","s521","s523"]  # left side$##################
side = "L"#######################

subject_dims_R = {  # subject-specific (sd, pdim)
    "s508": (3,2),
    "s514": (3,3),
    "s515": (3,4),
    "s517": (3,2),
    "s519": (5,2),
    "s520": (3,4),
    "s521": (5, 2),
    "s523": (5,4),
}
# neurips_plus_align_recover_F_L with help of var: 508:3,4/ 513:5,3/ 514:5,2/ 515: 5,4/ 518: 3,2/ 519:3,4/ 520:4,3/521: 5,2 /523: 5,4
subject_dims_L = {  # subject-specific (sd, pdim)
    "s508": (3,4),
    "s513": (5,3),
    "s514": (5,2),
    "s515": (5,4),
    "s518": (3,2),
    "s519": (3,4),
    "s520": (4,3),
    "s521": (5, 2),
    "s523": (5,4),
}
if side == "R":
    subject_dims = subject_dims_R
elif side=="L":
    subject_dims = subject_dims_L


seed_list = [702,703,704]

for rand_init_seed  in seed_list:
    # ------------------------
    # File paths: replace with your own subject/side paths
    # ------------------------
    for subject in subject_list:
        # subject = "s508"   # example
        # Latent dimensions: match your SPIRE grid
        shared_latent_dim, r1_specific_dim = subject_dims[subject]
        # r1_specific_dim = 4      # GPi private
        r2_specific_dim = r1_specific_dim      # STN private
        # shared_latent_dim = 3    # shared

        SUBJ_DIR = os.path.join(DATA_ROOT, subject)

        GPI_PATH = os.path.join(SUBJ_DIR, "gpi_train_off.pt")
        STN_PATH = os.path.join(SUBJ_DIR, "stn_train_off.pt")

        


        # ------------------------
        # Load tensors
        # Expected shape from your SPIRE pipeline: [N, W, C]
        # N = windows, W = timepoints, C = channels
        # ------------------------
        gpi = torch.load(GPI_PATH, map_location="cpu").float()
        stn = torch.load(STN_PATH, map_location="cpu").float()

        print("GPi:", gpi.shape)
        print("STN:", stn.shape)

        # Make sure same number of windows and timepoints
        N = min(gpi.shape[0], stn.shape[0])
        T = min(gpi.shape[1], stn.shape[1])

        gpi = gpi[:N, :T, :]
        stn = stn[:N, :T, :]

        # Optional but recommended: z-score per channel over all windows/time
        def zscore_lfp_tensor(x, eps=1e-8):
            # x: [N, T, C]
            mean = x.reshape(-1, x.shape[-1]).mean(dim=0)
            std = x.reshape(-1, x.shape[-1]).std(dim=0)
            return (x - mean) / (std + eps)

        gpi = zscore_lfp_tensor(gpi)
        stn = zscore_lfp_tensor(stn)

        data1 = gpi.numpy()
        data2 = stn.numpy()

        # CTAE expects concatenated regions along channel dimension: [N, T, Cgpi + Cstn]
        data = np.concatenate((data1, data2), axis=-1).astype(np.float32)

        input_dim1 = data1.shape[-1]
        input_dim2 = data2.shape[-1]
        num_neurons1 = input_dim1
        num_timeframes = data.shape[1]

        time = np.arange(num_timeframes) * bin_size

        print("Final CTAE input:", data.shape)
        print("GPi channels:", input_dim1)
        print("STN channels:", input_dim2)
        print("Timeframes:", num_timeframes)


        # ------------------------
        # Model save path
        # ------------------------
        model_path = (
            f"{TRAINED_MODELS_ROOT}/ctae_{subject}_{side}"
            f"_bs{batch_size}"
            f"_lr{safe_format(learning_rate)}"
            f"_L{num_layers}"
            f"_r1-{r1_specific_dim}_r2-{r2_specific_dim}"
            f"_s{shared_latent_dim}"
            f"_pe{'T' if pe else 'F'}"
            f"_align{safe_format(lambda_alignment)}"
            f"_ortho{safe_format(lambda_ortho)}"
            f"_recons2-{safe_format(lambda_recons2)}"
            f"_warm{warm_up_ortho}"
            f"_seed{rand_init_seed}"
            f"_ep{num_epochs}.pth"
        )
        print(f"\n=== {model_path} ===")

        hparam_dict = {
            "r1": r1_specific_dim,
            "r2": r2_specific_dim,
            "shared": shared_latent_dim,
            "nl": num_layers,
            "lambda_align": lambda_alignment,
            "lambda_ortho": lambda_ortho,
            "lr": learning_rate,
            "warm_up_ortho": warm_up_ortho,
            "batch_size": batch_size,
            "pe": pe,
        }

        hparam_str = "_".join([f"{k}-{safe_format(v)}" for k, v in hparam_dict.items()])
        print(model_path)

        train_loader, val_loader = create_train_val_loaders(
            data,
            batch_size=batch_size,
            train_ratio=0.8,
            shuffle=True,
            seed=0,
        )

        # ------------------------
        # Reproducibility
        # ------------------------

        torch.manual_seed(rand_init_seed)
        np.random.seed(rand_init_seed)
        random.seed(rand_init_seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed(rand_init_seed)
            torch.cuda.manual_seed_all(rand_init_seed)

        # ------------------------
        # Create model
        # ------------------------
        model = CoupledTransformerAutoencoderTwoRegions(
            input_dim1,
            input_dim2,
            r1_specific_dim,
            r2_specific_dim,
            shared_latent_dim,
            nhead,
            num_layers,
            num_layers,
            max_len,
            pe,
            pe_learn,
        )

        model = model.to(device)

        # ------------------------
        # Loss and optimizer
        # ------------------------
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        metadata = {
            "subject": subject,
            "side": side,
            "input_dim1": input_dim1,
            "input_dim2": input_dim2,
            "num_neurons1": num_neurons1,
            "num_timeframes": num_timeframes,
            "r1_specific_dim": r1_specific_dim,
            "r2_specific_dim": r2_specific_dim,
            "shared_latent_dim": shared_latent_dim,
            "nhead": nhead,
            "num_layers": num_layers,
            "max_len": max_len,
            "pe": pe,
            "pe_learn": pe_learn,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "num_epochs": num_epochs,
            "lambda_alignment": lambda_alignment,
            "lambda_ortho": lambda_ortho,
            "lambda_recons2": lambda_recons2,
            "warm_up_ortho": warm_up_ortho,
        }

        bundle_path = model_path.replace(".pth", "_bundle.pth")
        log_dir = model_path.replace(".pth", "_tb")

        history = train_ctae_with_logging(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=num_epochs,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            num_neurons1=num_neurons1,
            model_path=model_path,
            lambda_alignment=lambda_alignment,
            lambda_ortho=lambda_ortho,
            warm_up_ortho=warm_up_ortho,
            lambda_recons2=lambda_recons2,
            early_stopping=True,
            patience=30,
            min_delta=1e-4,
            start_epoch=80,
            log_dir=log_dir,
            bundle_path=bundle_path,
            metadata=metadata,
        )

GPi: torch.Size([480, 247, 72])
STN: torch.Size([480, 247, 12])
Final CTAE input: (480, 247, 84)
GPi channels: 72
STN channels: 12
Timeframes: 247

=== D:\copy_comp_project\ctae_models/ctae_s508_L_bs8_lr0.0001_L2_r1-4_r2-4_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
D:\copy_comp_project\ctae_models/ctae_s508_L_bs8_lr0.0001_L2_r1-4_r2-4_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth


c:\Users\rahil\Documents\ctae-sanger\models\ctae.py:110: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(


Saved best model | epoch 0 | val R2=-0.2511 | val loss=0.194963
Epoch 0000 | train loss=0.226994, val loss=0.194963, train R2=-0.4335, val R2=-0.2511, align=0.008212, ortho=3.341446
Saved best model | epoch 1 | val R2=-0.1982 | val loss=0.190127
Saved best model | epoch 2 | val R2=-0.1507 | val loss=0.185677
Saved best model | epoch 3 | val R2=-0.1021 | val loss=0.180954
Saved best model | epoch 4 | val R2=-0.0484 | val loss=0.175830
Saved best model | epoch 5 | val R2=-0.0131 | val loss=0.172755
Saved best model | epoch 6 | val R2=-0.0061 | val loss=0.170748
Saved best model | epoch 7 | val R2=0.0227 | val loss=0.167725
Saved best model | epoch 8 | val R2=0.0376 | val loss=0.165530
Saved best model | epoch 9 | val R2=0.0477 | val loss=0.164278
Saved best model | epoch 10 | val R2=0.0619 | val loss=0.163429
Epoch 0010 | train loss=0.181360, val loss=0.163429, train R2=-0.0040, val R2=0.0619, align=0.044817, ortho=5.523430
Saved best model | epoch 12 | val R2=0.0721 | val loss=0.161171


In [5]:
# if os.path.exists(model_path):
#     print(f"Model already exists: {model_path}")
# else:
DATA_ROOT = r"D:\copy_comp_project\Off_tensor_Data_R"#################
TRAINED_MODELS_ROOT = r"D:\copy_comp_project\ctae_models"
os.makedirs(TRAINED_MODELS_ROOT, exist_ok=True)
# subject_list = ["s508", "s513","s514", "s515","s518","s519","s520","s521","s523"]  # left side$##################
subject_list = ["s508","s514", "s515","s517","s519","s520","s521","s523"]  # right side
side = "R"#######################

subject_dims_R = {  # subject-specific (sd, pdim)
    "s508": (3,2),
    "s514": (3,3),
    "s515": (3,4),
    "s517": (3,2),
    "s519": (5,2),
    "s520": (3,4),
    "s521": (5, 2),
    "s523": (5,4),
}
# neurips_plus_align_recover_F_L with help of var: 508:3,4/ 513:5,3/ 514:5,2/ 515: 5,4/ 518: 3,2/ 519:3,4/ 520:4,3/521: 5,2 /523: 5,4
subject_dims_L = {  # subject-specific (sd, pdim)
    "s508": (3,4),
    "s513": (5,3),
    "s514": (5,2),
    "s515": (5,4),
    "s518": (3,2),
    "s519": (3,4),
    "s520": (4,3),
    "s521": (5, 2),
    "s523": (5,4),
}
if side == "R":
    subject_dims = subject_dims_R
elif side=="L":
    subject_dims = subject_dims_L


seed_list = [702,703,704]

for rand_init_seed  in seed_list:
    # ------------------------
    # File paths: replace with your own subject/side paths
    # ------------------------
    for subject in subject_list:
        # subject = "s508"   # example
        # Latent dimensions: match your SPIRE grid
        shared_latent_dim, r1_specific_dim = subject_dims[subject]
        # r1_specific_dim = 4      # GPi private
        r2_specific_dim = r1_specific_dim      # STN private
        # shared_latent_dim = 3    # shared

        SUBJ_DIR = os.path.join(DATA_ROOT, subject)

        GPI_PATH = os.path.join(SUBJ_DIR, "gpi_train_off.pt")
        STN_PATH = os.path.join(SUBJ_DIR, "stn_train_off.pt")

        


        # ------------------------
        # Load tensors
        # Expected shape from your SPIRE pipeline: [N, W, C]
        # N = windows, W = timepoints, C = channels
        # ------------------------
        gpi = torch.load(GPI_PATH, map_location="cpu").float()
        stn = torch.load(STN_PATH, map_location="cpu").float()

        print("GPi:", gpi.shape)
        print("STN:", stn.shape)

        # Make sure same number of windows and timepoints
        N = min(gpi.shape[0], stn.shape[0])
        T = min(gpi.shape[1], stn.shape[1])

        gpi = gpi[:N, :T, :]
        stn = stn[:N, :T, :]

        # Optional but recommended: z-score per channel over all windows/time
        def zscore_lfp_tensor(x, eps=1e-8):
            # x: [N, T, C]
            mean = x.reshape(-1, x.shape[-1]).mean(dim=0)
            std = x.reshape(-1, x.shape[-1]).std(dim=0)
            return (x - mean) / (std + eps)

        gpi = zscore_lfp_tensor(gpi)
        stn = zscore_lfp_tensor(stn)

        data1 = gpi.numpy()
        data2 = stn.numpy()

        # CTAE expects concatenated regions along channel dimension: [N, T, Cgpi + Cstn]
        data = np.concatenate((data1, data2), axis=-1).astype(np.float32)

        input_dim1 = data1.shape[-1]
        input_dim2 = data2.shape[-1]
        num_neurons1 = input_dim1
        num_timeframes = data.shape[1]

        time = np.arange(num_timeframes) * bin_size

        print("Final CTAE input:", data.shape)
        print("GPi channels:", input_dim1)
        print("STN channels:", input_dim2)
        print("Timeframes:", num_timeframes)


        # ------------------------
        # Model save path
        # ------------------------
        model_path = (
            f"{TRAINED_MODELS_ROOT}/ctae_{subject}_{side}"
            f"_bs{batch_size}"
            f"_lr{safe_format(learning_rate)}"
            f"_L{num_layers}"
            f"_r1-{r1_specific_dim}_r2-{r2_specific_dim}"
            f"_s{shared_latent_dim}"
            f"_pe{'T' if pe else 'F'}"
            f"_align{safe_format(lambda_alignment)}"
            f"_ortho{safe_format(lambda_ortho)}"
            f"_recons2-{safe_format(lambda_recons2)}"
            f"_warm{warm_up_ortho}"
            f"_seed{rand_init_seed}"
            f"_ep{num_epochs}.pth"
        )
        print(f"\n=== {model_path} ===")

        hparam_dict = {
            "r1": r1_specific_dim,
            "r2": r2_specific_dim,
            "shared": shared_latent_dim,
            "nl": num_layers,
            "lambda_align": lambda_alignment,
            "lambda_ortho": lambda_ortho,
            "lr": learning_rate,
            "warm_up_ortho": warm_up_ortho,
            "batch_size": batch_size,
            "pe": pe,
        }

        hparam_str = "_".join([f"{k}-{safe_format(v)}" for k, v in hparam_dict.items()])
        print(model_path)

        train_loader, val_loader = create_train_val_loaders(
            data,
            batch_size=batch_size,
            train_ratio=0.8,
            shuffle=True,
            seed=0,
        )

        # ------------------------
        # Reproducibility
        # ------------------------

        torch.manual_seed(rand_init_seed)
        np.random.seed(rand_init_seed)
        random.seed(rand_init_seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed(rand_init_seed)
            torch.cuda.manual_seed_all(rand_init_seed)

        # ------------------------
        # Create model
        # ------------------------
        model = CoupledTransformerAutoencoderTwoRegions(
            input_dim1,
            input_dim2,
            r1_specific_dim,
            r2_specific_dim,
            shared_latent_dim,
            nhead,
            num_layers,
            num_layers,
            max_len,
            pe,
            pe_learn,
        )

        model = model.to(device)

        # ------------------------
        # Loss and optimizer
        # ------------------------
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        metadata = {
            "subject": subject,
            "side": side,
            "input_dim1": input_dim1,
            "input_dim2": input_dim2,
            "num_neurons1": num_neurons1,
            "num_timeframes": num_timeframes,
            "r1_specific_dim": r1_specific_dim,
            "r2_specific_dim": r2_specific_dim,
            "shared_latent_dim": shared_latent_dim,
            "nhead": nhead,
            "num_layers": num_layers,
            "max_len": max_len,
            "pe": pe,
            "pe_learn": pe_learn,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "num_epochs": num_epochs,
            "lambda_alignment": lambda_alignment,
            "lambda_ortho": lambda_ortho,
            "lambda_recons2": lambda_recons2,
            "warm_up_ortho": warm_up_ortho,
        }

        bundle_path = model_path.replace(".pth", "_bundle.pth")
        log_dir = model_path.replace(".pth", "_tb")

        history = train_ctae_with_logging(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=num_epochs,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            num_neurons1=num_neurons1,
            model_path=model_path,
            lambda_alignment=lambda_alignment,
            lambda_ortho=lambda_ortho,
            warm_up_ortho=warm_up_ortho,
            lambda_recons2=lambda_recons2,
            early_stopping=True,
            patience=30,
            min_delta=1e-4,
            start_epoch=80,
            log_dir=log_dir,
            bundle_path=bundle_path,
            metadata=metadata,
        )

GPi: torch.Size([480, 247, 48])
STN: torch.Size([480, 247, 24])
Final CTAE input: (480, 247, 72)
GPi channels: 48
STN channels: 24
Timeframes: 247

=== D:\copy_comp_project\ctae_models/ctae_s508_R_bs8_lr0.0001_L2_r1-2_r2-2_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
D:\copy_comp_project\ctae_models/ctae_s508_R_bs8_lr0.0001_L2_r1-2_r2-2_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth
Saved best model | epoch 0 | val R2=-0.8451 | val loss=0.140893
Epoch 0000 | train loss=0.165492, val loss=0.140893, train R2=-1.1396, val R2=-0.8451, align=0.013463, ortho=2.278850
Saved best model | epoch 1 | val R2=-0.6930 | val loss=0.135486
Saved best model | epoch 2 | val R2=-0.6079 | val loss=0.132152
Saved best model | epoch 3 | val R2=-0.5235 | val loss=0.129002
Saved best model | epoch 4 | val R2=-0.4463 | val loss=0.126118
Saved best model | epoch 5 | val R2=-0.3674 | val loss=0.123170
Saved best model | epoch 6 | val R2=-0.3196 | val loss=0.121505
Saved best

In [ ]:
# ### loadng the model:
# # bundle = torch.load(bundle_path, map_location=device, weights_only=False)

# # model = bundle["model"]
# # model = model.to(device)
# # model.eval()

# # metadata = bundle["metadata"]
# # history = bundle["history"]

# #orrr
# bundle = torch.load(bundle_path, map_location=device, weights_only=False)
# metadata = bundle["metadata"]

# model = CoupledTransformerAutoencoderTwoRegions(
#     metadata["input_dim1"],
#     metadata["input_dim2"],
#     metadata["r1_specific_dim"],
#     metadata["r2_specific_dim"],
#     metadata["shared_latent_dim"],
#     metadata["nhead"],
#     metadata["num_layers"],
#     metadata["num_layers"],
#     metadata["max_len"],
#     metadata["pe"],
#     metadata["pe_learn"],
# ).to(device)

# model.load_state_dict(bundle["model_state_dict"])
# model.eval()

: 